# mutascape validation: GB1 small-site complete combinatorial

Validate mutascape against the Wu et al. 2016 GB1 dataset: 4 positions (V39/D40/G41/V54), 149,360 of 160,000 possible variants measured ( ~93.4% coverage).

Unlike the sparse random sampling in the GFP validation, GB1's near-complete space lets us compute ground-truth statistics that are otherwise impossible:
* Walsh–Hadamard variance decomposition (Poelwijk et al. 2019)
* Exact local optima counts and basin sizes (Szendro et al. 2013)
* Monotonic accessible-path fraction to the global peak (Weinreich et al. 2006)
* Reciprocal sign epistasis (Poelwijk 2011)

We calibrate mutascape's alpha on the same low-order epistasis targets as the GFP validation, then check whether the calibrated landscape reproduces these held-out complete-space properties.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd

from mutascape.validation.calibration import (
    alpha_sweep,
    build_landscape,
    find_optimal_alpha,
    refine_alpha,
)
from mutascape.validation.comparisons import (
    composite_validation_score,
    fitness_wasserstein,
    tensor_pearson_r,
)
from mutascape.validation.figures import (
    fig_alpha_calibration,
    fig_fitness_distribution,
    fig_sigmoid_curves,
    fig_tensor_scatter,
    fig_validation_summary,
    fig_walsh_spectrum,
    save_fig,
)
from mutascape.validation.fourier import (
    fitness_tensor_from_landscape,
    fitness_tensor_from_real,
    observed_alphabets,
    variance_fraction_comparison,
    walsh_decompose,
    walsh_l1_distance,
)
from mutascape.validation.metrics import (
    accessible_paths,
    compute_all_metrics,
    fitness_distance_correlation,
    fitness_distance_correlation_from_tensor,
    local_optima_from_tensor,
    roughness_slope_ratio,
)
from mutascape.validation.real_data import (
    load_gb1_combinatorial,
    load_single_site_effects,
    sample_synthetic_data,
)

DATA_DIR = Path("../../data/gb1")
OUTPUT_DIR = Path("./output_gb1")
OUTPUT_DIR.mkdir(exist_ok=True)

DMS_CSV = DATA_DIR / "gb1.csv"
COMBINATORIAL_CSV = DATA_DIR / "gb1_combinatorial.csv"
DISTANCES_NPZ = DATA_DIR / "gb1_distances.npz"
WT_FASTA = DATA_DIR / "gb1_wt.fasta"

POSITIONS = (39, 40, 41, 54)
WT_AA_AT_POSITION = {39: "V", 40: "D", 41: "G", 54: "V"}
DEPTHS = [2, 3, 4]
N_PER_DEPTH = 3000
SEED = 42


def make_landscape(alpha, sigma=0.3, model="random"):
    return build_landscape(
        DMS_CSV,
        DISTANCES_NPZ,
        WT_FASTA,
        alpha=alpha,
        sigma=sigma,
        seed=SEED,
        epistasis_model=model,
    )

## Load real GB1 combinatorial data

In [2]:
real_df = load_gb1_combinatorial(
    COMBINATORIAL_CSV,
    DMS_CSV,
    positions=POSITIONS,
    min_depth=1,
    max_depth=4,
)
single_site_effects = load_single_site_effects(DMS_CSV)
print(f"n variants: {len(real_df)}")
print(f"fitness range: [{real_df['fitness'].min():.3f}, {real_df['fitness'].max():.3f}]")
print(f"mean epistatic deviation (|resid|): {real_df['epistatic_deviation'].abs().mean():.3f}")

Loaded 149360/149360 variants (0 skipped, missing single-site effects)
Depth distribution: {1: 76, 2: 2091, 3: 26019, 4: 121174}
n variants: 149360
fitness range: [-1.000, 7.762]
mean epistatic deviation (|resid|): 1.070


## Real landscape metrics (low-order)

In [3]:
real_metrics = compute_all_metrics(real_df, single_site_effects, DEPTHS)
print(
    "additive correlation by depth:",
    {d: f"{v:.3f}" for d, v in real_metrics["additive_correlation"].items()},
)
print(
    "sign epistasis by depth:", {d: f"{v:.3f}" for d, v in real_metrics["sign_epistasis"].items()}
)
print(f"r/s ratio: {roughness_slope_ratio(real_df):.3f}")
print(f"fitness-distance correlation (depth): {fitness_distance_correlation(real_df):.3f}")

additive correlation by depth: {2: '0.589', 3: '0.326', 4: '0.141'}
sign epistasis by depth: {2: '0.305', 3: '0.442', 4: '0.658'}
r/s ratio: 0.807
fitness-distance correlation (depth): -0.112


## Coarse alpha sweep

In [4]:
coarse = alpha_sweep(
    real_metrics,
    single_site_effects,
    make_landscape,
    alphas=[0.0, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 12.0],
    depths=DEPTHS,
    n_per_depth=N_PER_DEPTH,
)
coarse_best = find_optimal_alpha(coarse)
print(f"coarse best alpha: {coarse_best}")
coarse

  alpha=0.00

 -> score=0.6175
  alpha=0.25

 -> score=0.5516
  alpha=0.50

 -> score=0.4774
  alpha=1.00

 -> score=0.3276
  alpha=2.00

 -> score=0.2711
  alpha=3.00

 -> score=0.3731
  alpha=5.00

 -> score=0.6588
  alpha=8.00

 -> score=1.1741
  alpha=12.00

 -> score=1.8675
coarse best alpha: 2.0


,alpha,composite_score,additive_corr_d2,sign_epi_d2,additive_corr_d3,sign_epi_d3,additive_corr_d4,sign_epi_d4
0,0.00,0.617464,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
1,0.25,0.551599,0.996107,0.016000,0.991223,0.088667,0.982609,0.160667
2,0.50,0.477367,0.984974,0.051000,0.967083,0.179667,0.934709,0.324000
3,1.00,0.327629,0.945544,0.136000,0.889996,0.361333,0.792401,0.553333
4,2.00,0.271111,0.833042,0.281333,0.716651,0.569333,0.535191,0.734667
5,3.00,0.373143,0.720652,0.365000,0.584009,0.658333,0.380736,0.795667
6,5.00,0.658842,0.554521,0.432000,0.427552,0.734000,0.228116,0.855667
7,8.00,1.174095,0.415266,0.501000,0.317062,0.787333,0.133275,0.897333
8,12.00,1.867545,0.321811,0.532333,0.249169,0.812000,0.078775,0.919333


In [5]:
fine = refine_alpha(
    real_metrics,
    single_site_effects,
    make_landscape,
    coarse_best=coarse_best,
    step=0.25,
    half_range=1.5,
    depths=DEPTHS,
    n_per_depth=N_PER_DEPTH,
)
best_alpha = find_optimal_alpha(fine)
print(f"best alpha (refined): {best_alpha}")
fine

  alpha=0.50

 -> score=0.4774
  alpha=0.75

 -> score=0.4011
  alpha=1.00

 -> score=0.3276
  alpha=1.25

 -> score=0.2757
  alpha=1.50

 -> score=0.2529
  alpha=1.75

... (5 outputs left out)

 -> score=0.3731
  alpha=3.25

 -> score=0.4048
  alpha=3.50

 -> score=0.4383
best alpha (refined): 1.5


,alpha,composite_score,additive_corr_d2,sign_epi_d2,additive_corr_d3,sign_epi_d3,additive_corr_d4,sign_epi_d4
0,0.50,0.477367,0.984974,0.051000,0.967083,0.179667,0.934709,0.324000
1,0.75,0.401055,0.967688,0.084000,0.931846,0.273667,0.867341,0.463667
2,1.00,0.327629,0.945544,0.136000,0.889996,0.361333,0.792401,0.553333
3,1.25,0.275733,0.919872,0.180333,0.845309,0.433667,0.718434,0.617333
4,1.50,0.252914,0.891899,0.222667,0.800503,0.493000,0.650019,0.667333
5,1.75,0.255540,0.862675,0.253667,0.757300,0.535000,0.588888,0.706000
6,2.00,0.271111,0.833042,0.281333,0.716651,0.569333,0.535191,0.734667
7,2.25,0.290818,0.803636,0.305333,0.678984,0.598333,0.488354,0.755000
8,2.50,0.315176,0.774916,0.336667,0.644398,0.620333,0.447552,0.768667
9,2.75,0.343265,0.747190,0.352667,0.612803,0.642000,0.411938,0.783667


In [6]:
full_sweep = (
    pd.concat([coarse, fine], ignore_index=True).sort_values("alpha").reset_index(drop=True)
)
fig = fig_alpha_calibration(full_sweep)
save_fig(fig, "gb1_alpha_calibration", OUTPUT_DIR)

  Saved gb1_alpha_calibration.pdf/.png


## Synthetic landscape at best alpha: low-order match

In [7]:
best_landscape = make_landscape(best_alpha)
synth_df = sample_synthetic_data(best_landscape, DEPTHS, n_per_depth=N_PER_DEPTH)
synth_metrics = compute_all_metrics(synth_df, single_site_effects, DEPTHS)

score = composite_validation_score(real_metrics, synth_metrics)
print(f"composite score at alpha={best_alpha}: {score:.4f}")
for d in DEPTHS:
    real_c = real_metrics["additive_correlation"].get(d)
    synth_c = synth_metrics["additive_correlation"].get(d)
    real_s = real_metrics["sign_epistasis"].get(d)
    synth_s = synth_metrics["sign_epistasis"].get(d)
    print(
        f"depth {d}: add_corr real={real_c:.3f} synth={synth_c:.3f} | "
        f"sign_epi real={real_s:.3f} synth={synth_s:.3f}"
    )

fig = fig_validation_summary(
    real_metrics,
    synth_metrics,
    best_alpha=best_alpha,
    depths=DEPTHS,
    protein_name="GB1",
)
save_fig(fig, "gb1_validation_summary", OUTPUT_DIR)

composite score at alpha=1.5: 0.2529
depth 2: add_corr real=0.589 synth=0.892 | sign_epi real=0.305 synth=0.223
depth 3: add_corr real=0.326 synth=0.801 | sign_epi real=0.442 synth=0.493
depth 4: add_corr real=0.141 synth=0.650 | sign_epi real=0.658 synth=0.667


  Saved gb1_validation_summary.pdf/.png


## Complete-space statistics: fitness tensors

The next cells build 20x20x20x20 fitness tensors for both real and synthetic landscapes and compute held-out structural properties that were not part of the alpha calibration objective.

In [8]:
alphabets = observed_alphabets(real_df, POSITIONS, WT_AA_AT_POSITION)
for p, a in alphabets.items():
    print(f"pos {p} ({WT_AA_AT_POSITION[p]}): {len(a)} AAs observed")

real_tensor, real_coverage = fitness_tensor_from_real(
    real_df, POSITIONS, WT_AA_AT_POSITION, alphabets
)
synth_tensor = fitness_tensor_from_landscape(best_landscape, POSITIONS, alphabets)
print(f"real tensor: shape {real_tensor.shape}, coverage {real_coverage:.3f}")
print(f"synth tensor: shape {synth_tensor.shape}")

pos 39 (V): 20 AAs observed
pos 40 (D): 20 AAs observed
pos 41 (G): 20 AAs observed
pos 54 (V): 20 AAs observed


real tensor: shape (20, 20, 20, 20), coverage 0.934
synth tensor: shape (20, 20, 20, 20)


### Walsh-Hadamard variance decomposition

For a small complete combinatorial space we can split total fitness variance *exactly* into contributions from each **order of interaction**:

- **Order 1** -- variance explained by treating each position independently. "On average across all backgrounds, swapping V39 to A shifts fitness by x." Everything a purely additive model can capture.
- **Order 2** -- extra variance that emerges only when two mutations co-occur, beyond what single-site averages predict. "V39A and V54G together give a fitness bump that neither single-site effect predicts alone."
- **Orders 3, 4** -- surprises requiring a specific trio or quad of mutations. Context-dependent effects that no lower-order model can reproduce.

Mathematically this is the protein-fitness analog of ANOVA on a complete factorial experiment (Poelwijk et al. 2019, *Nat Commun* 10:4213). The orders are orthogonal and their variance fractions sum to 1. The decomposition is only well-defined on a complete (or near-complete) space -- you cannot compute it from sparse random samples like GFP's.

**Why it matters here.** mutascape's fitness function is

$$f(x) = \sum_i h_i(x_i) + \alpha \sum_{i<j} C_{ij}\,J_{ij}(x_i, x_j) + \text{noise}$$

which contains only single-site terms ($h$) and pairwise couplings ($J$). By construction, it can only carry variance at Walsh orders 1 and 2. Any 3rd- or 4th-order Walsh fraction observed in real GB1 is therefore a hard *unreachable* upper bound for the current model -- not something $\alpha$ can fix. The comparison below is both a measure of how well mutascape reproduces GB1's pairwise structure and a diagnostic of where the pairwise-only assumption runs out.

In [9]:
walsh = variance_fraction_comparison(real_tensor, synth_tensor)
print("                 order 1   order 2   order 3   order 4")
for label, name in [("real", "real"), ("synth", "synth")]:
    row = "  ".join(f"{walsh[name].get(k, 0):.4f}" for k in sorted(walsh[name]))
    print(f"{label:>14} : {row}")
print(f"max |real - synth| across orders: {walsh['max_abs_diff']:.4f}")
print(f"mean |real - synth| across orders: {walsh['mean_abs_diff']:.4f}")

                 order 1   order 2   order 3   order 4
          real : 0.1856  0.4293  0.3084  0.0767
         synth : 0.8413  0.1587  0.0000  0.0000
max |real - synth| across orders: 0.6557
mean |real - synth| across orders: 0.3278


### Local optima

In [10]:
real_opt = local_optima_from_tensor(real_tensor)
synth_opt = local_optima_from_tensor(synth_tensor)
print(f"real local optima: n={real_opt['n_optima']}")
optima_ratio = synth_opt["n_optima"] / max(real_opt["n_optima"], 1)
print(f"synth local optima: n={synth_opt['n_optima']} (ratio synth/real = {optima_ratio:.2f})")
print(
    "real top-5 peaks:",
    [
        (idx, f"{f:.3f}")
        for idx, f in zip(real_opt["peak_indices"][:5], real_opt["peak_fitnesses"][:5])
    ],
)
print(
    "synth top-5 peaks:",
    [
        (idx, f"{f:.3f}")
        for idx, f in zip(synth_opt["peak_indices"][:5], synth_opt["peak_fitnesses"][:5])
    ],
)
print(f"real basin sizes top-5: {sorted(real_opt['basin_sizes'], reverse=True)[:5]}")
print(f"synth basin sizes top-5: {sorted(synth_opt['basin_sizes'], reverse=True)[:5]}")

real local optima: n=182
synth local optima: n=5 (ratio synth/real = 0.03)
real top-5 peaks: [((np.int64(4), np.int64(18), np.int64(0), np.int64(0)), '7.762'), ((np.int64(0), np.int64(11), np.int64(1), np.int64(0)), '6.557'), ((np.int64(4), np.int64(18), np.int64(9), np.int64(5)), '6.313'), ((np.int64(17), np.int64(0), np.int64(0), np.int64(0)), '5.201'), ((np.int64(7), np.int64(18), np.int64(5), np.int64(4)), '4.772')]
synth top-5 peaks: [((np.int64(18), np.int64(18), np.int64(12), np.int64(1)), '6.639'), ((np.int64(2), np.int64(18), np.int64(12), np.int64(2)), '6.587'), ((np.int64(7), np.int64(18), np.int64(3), np.int64(3)), '5.945'), ((np.int64(10), np.int64(18), np.int64(19), np.int64(19)), '5.832'), ((np.int64(15), np.int64(18), np.int64(3), np.int64(16)), '5.444')]
real basin sizes top-5: [40721, 32272, 25131, 9579, 7847]
synth basin sizes top-5: [79090, 50370, 20335, 6629, 3576]


### Accessible pathways and reciprocal sign epistasis

In [11]:
real_paths = accessible_paths(real_tensor, n_sources=5000, n_rsign_samples=40000, seed=SEED)
synth_paths = accessible_paths(synth_tensor, n_sources=5000, n_rsign_samples=40000, seed=SEED)
print(
    f"real: accessible fraction = {real_paths['mean_accessible_fraction']:.4f}, "
    f"RSE fraction = {real_paths['reciprocal_sign_frac']:.4f}"
)
print(
    f"synth: accessible fraction = {synth_paths['mean_accessible_fraction']:.4f}, "
    f"RSE fraction = {synth_paths['reciprocal_sign_frac']:.4f}"
)
real_fdc = fitness_distance_correlation_from_tensor(real_tensor)
synth_fdc = fitness_distance_correlation_from_tensor(synth_tensor)
print(f"real FDC (tensor, distance to peak): {real_fdc:.4f}")
print(f"synth FDC (tensor, distance to peak): {synth_fdc:.4f}")

real:  accessible fraction = 0.3168, RSE fraction = 0.1521
synth: accessible fraction = 0.7662, RSE fraction = 0.0362
real FDC (tensor, distance to peak): -0.1685
synth FDC (tensor, distance to peak): -0.3723


## Summary table of validation criteria

In [12]:
summary = pd.DataFrame(
    [
        {"metric": "Walsh order 1", "real": walsh["real"][1], "synth": walsh["synth"][1]},
        {"metric": "Walsh order 2", "real": walsh["real"][2], "synth": walsh["synth"][2]},
        {"metric": "Walsh order 3", "real": walsh["real"][3], "synth": walsh["synth"][3]},
        {"metric": "Walsh order 4", "real": walsh["real"][4], "synth": walsh["synth"][4]},
        {"metric": "local optima", "real": real_opt["n_optima"], "synth": synth_opt["n_optima"]},
        {
            "metric": "accessible-path fraction",
            "real": real_paths["mean_accessible_fraction"],
            "synth": synth_paths["mean_accessible_fraction"],
        },
        {
            "metric": "reciprocal sign epi",
            "real": real_paths["reciprocal_sign_frac"],
            "synth": synth_paths["reciprocal_sign_frac"],
        },
        {
            "metric": "r/s ratio",
            "real": roughness_slope_ratio(real_df),
            "synth": roughness_slope_ratio(synth_df),
        },
        {
            "metric": "FDC (tensor, to peak)",
            "real": fitness_distance_correlation_from_tensor(real_tensor),
            "synth": fitness_distance_correlation_from_tensor(synth_tensor),
        },
    ]
)
summary["abs_diff"] = (summary["synth"] - summary["real"]).abs()
summary.to_csv(OUTPUT_DIR / "gb1_summary.csv", index=False)
summary

,metric,real,synth,abs_diff
0,Walsh order 1,0.185603,8.413025e-01,0.655699
1,Walsh order 2,0.429308,1.586975e-01,0.270610
2,Walsh order 3,0.308366,6.449954e-30,0.308366
3,Walsh order 4,0.076723,2.394260e-30,0.076723
4,local optima,182.000000,5.000000e+00,177.000000
5,accessible-path fraction,0.316758,7.662116e-01,0.449453
6,reciprocal sign epi,0.152125,3.622500e-02,0.115900
7,r/s ratio,0.806941,7.889075e-01,0.018033
8,"FDC (tensor, to peak)",-0.168519,-3.722989e-01,0.203780


## Ablation: RandomEpistasis vs SignConsistentEpistasis

In [13]:
ablation_rows = []
for model in ["random", "sign_consistent"]:
    ls = make_landscape(best_alpha, model=model)
    synth_t = fitness_tensor_from_landscape(ls, POSITIONS, alphabets)
    w = walsh_decompose(synth_t)
    opt = local_optima_from_tensor(synth_t)
    paths = accessible_paths(synth_t, n_sources=2000, n_rsign_samples=20000, seed=SEED)
    ablation_rows.append(
        {
            "model": model,
            "walsh_1": w[1],
            "walsh_2": w[2],
            "walsh_3": w[3],
            "walsh_4": w[4],
            "n_optima": opt["n_optima"],
            "accessible_fraction": paths["mean_accessible_fraction"],
            "rse_fraction": paths["reciprocal_sign_frac"],
        }
    )
ablation = pd.DataFrame(ablation_rows)
ablation.to_csv(OUTPUT_DIR / "gb1_ablation.csv", index=False)
ablation

,model,walsh_1,walsh_2,walsh_3,walsh_4,n_optima,accessible_fraction,rse_fraction
0,random,0.841302,0.158698,6.449954e-30,2.394260e-30,5,0.765688,0.03485
1,sign_consistent,0.933709,0.066291,3.243419e-29,1.214364e-29,3,0.903958,0.02555


## Thermodynamic non-linearity layer

The analysis above showed mutascape with a pairwise-only model cannot reach Walsh orders 3-4 (both zero by construction) or match real GB1's dense local-optima structure. This section evaluates the proposed fix: wrap the fitness function in a global non-linearity $g$ whose parameters are fit from the single-site DMS distribution alone. See `docs/nonlinearity_design.md` for the derivation and `docs/references.md` for the literature (Otwinowski 2018, Sailer & Harms 2017, Nisthal 2019).

**The key distinction for singles-only calibration.** Floor and scale are well identified from the observed DFE. **Ceiling is not.** GB1's singles max out at +2.9 but combinatorials reach +7.8, so the singles-only auto-ceiling truncates the landscape. For proteins without combinatorial DMS we introduce `combinatorial_upside`: a multiplier on the positive singles range that captures the expected fitness headroom from stacking stabilising mutations. `combinatorial_upside=1` is the naive singles-only ceiling, `=3` is a reasonable prior for stability-limited proteins.

In [14]:
from mutascape.nonlinearity import GlobalEpistasisSigmoid
from mutascape.validation.calibration import find_optimal_thermo, thermo_sweep

# Fit a baseline sigmoid from singles alone (no combinatorial knowledge)
g_singles = GlobalEpistasisSigmoid.fit_from_singles(single_site_effects)
print(
    f"Singles-only fit: floor={g_singles.floor:.3f}, "
    f"ceiling={g_singles.ceiling:.3f}, scale={g_singles.scale:.3f}"
)

# Same with combinatorial_upside=3 (stability-stacking prior for proteins
# without combinatorial DMS)
g_upside = GlobalEpistasisSigmoid.fit_from_singles(single_site_effects, combinatorial_upside=3.0)
print(
    f"Upside=3 fit: floor={g_upside.floor:.3f}, "
    f"ceiling={g_upside.ceiling:.3f}, scale={g_upside.scale:.3f}"
)

Singles-only fit: floor=-1.191, ceiling=3.096, scale=0.595
Upside=3 fit:   floor=-1.191, ceiling=8.899, scale=0.708


In [15]:
def build_thermo(alpha, nl):
    return build_landscape(
        DMS_CSV,
        DISTANCES_NPZ,
        WT_FASTA,
        alpha=alpha,
        seed=SEED,
        nonlinearity=nl,
    )


# Evaluate each non-linearity configuration at alpha=1.5 (the Identity-calibrated best).
# The three held-out scalars (tensor_r, walsh_L1, fitness_W1) compare each synthetic
# configuration against the real GB1 tensor and Walsh spectrum computed earlier.
def eval_at(label, alpha, nl):
    ls = build_thermo(alpha, nl)
    synth_df = sample_synthetic_data(ls, DEPTHS, n_per_depth=N_PER_DEPTH)
    m = compute_all_metrics(synth_df, single_site_effects, DEPTHS)
    t = fitness_tensor_from_landscape(ls, POSITIONS, alphabets)
    w = walsh_decompose(t)
    o = local_optima_from_tensor(t)
    p = accessible_paths(t, n_sources=3000, n_rsign_samples=20000, seed=SEED)
    return {
        "config": label,
        "alpha": alpha,
        "add_corr_d2": m["additive_correlation"].get(2, float("nan")),
        "add_corr_d4": m["additive_correlation"].get(4, float("nan")),
        "walsh_1": w[1],
        "walsh_2": w[2],
        "walsh_3": w[3],
        "walsh_4": w[4],
        "n_optima": o["n_optima"],
        "accessible_frac": p["mean_accessible_fraction"],
        "rs_ratio": roughness_slope_ratio(synth_df),
        "fdc": fitness_distance_correlation_from_tensor(t),
        "tensor_r": tensor_pearson_r(real_tensor, t),
        "walsh_L1": walsh_l1_distance(walsh["real"], w),
        "fitness_W1": fitness_wasserstein(real_tensor, t),
    }


thermo_rows = [
    {
        "config": "REAL GB1",
        "alpha": None,
        "add_corr_d2": real_metrics["additive_correlation"].get(2),
        "add_corr_d4": real_metrics["additive_correlation"].get(4),
        "walsh_1": walsh["real"][1],
        "walsh_2": walsh["real"][2],
        "walsh_3": walsh["real"][3],
        "walsh_4": walsh["real"][4],
        "n_optima": real_opt["n_optima"],
        "accessible_frac": real_paths["mean_accessible_fraction"],
        "rs_ratio": roughness_slope_ratio(real_df),
        "fdc": fitness_distance_correlation_from_tensor(real_tensor),
        "tensor_r": float("nan"),
        "walsh_L1": float("nan"),
        "fitness_W1": float("nan"),
    },
    eval_at("Identity (baseline)", 1.5, None),
    eval_at("Thermo, upside=1 (singles-only)", 1.5, g_singles),
    eval_at("Thermo, upside=3 (stability prior)", 1.5, g_upside),
]
thermo_cmp = pd.DataFrame(thermo_rows)
thermo_cmp.to_csv(OUTPUT_DIR / "gb1_thermo_comparison.csv", index=False)
thermo_cmp.round(3)

,config,alpha,add_corr_d2,add_corr_d4,walsh_1,walsh_2,walsh_3,walsh_4,n_optima,accessible_frac,rs_ratio,fdc,tensor_r,walsh_L1,fitness_W1
0,REAL GB1,NaN,0.589,0.141,0.186,0.429,0.308,0.077,182,0.317,0.807,-0.169,NaN,NaN,NaN
1,Identity (baseline),1.5,0.892,0.650,0.841,0.159,0.000,0.000,5,0.772,0.789,-0.372,0.104,1.311,1.323
2,"Thermo, upside=1 (singles-only)",1.5,0.852,0.539,0.668,0.238,0.073,0.021,8,0.738,0.772,-0.387,0.128,0.964,0.547
3,"Thermo, upside=3 (stability prior)",1.5,0.727,0.359,0.457,0.382,0.137,0.024,9,0.729,0.874,-0.322,0.143,0.543,0.417


### Joint (alpha, scale) calibration under the non-linearity

With g in the model, the alpha optimum shifts. The pairwise-only calibration picked alpha=1.5 against an Identity g; re-running the composite sweep over both alpha and the sigmoid scale finds the best joint point. Ceiling is held at the `combinatorial_upside=3` prior since that is the setting a no-DMS user would have.

In [16]:
# Hold floor and ceiling at the upside=3 fit; sweep alpha and scale.
_base = GlobalEpistasisSigmoid.fit_from_singles(single_site_effects, combinatorial_upside=3.0)


def make_thermo(alpha, scale):
    g = GlobalEpistasisSigmoid(floor=_base.floor, ceiling=_base.ceiling, scale=scale)
    return build_landscape(
        DMS_CSV,
        DISTANCES_NPZ,
        WT_FASTA,
        alpha=alpha,
        seed=SEED,
        nonlinearity=g,
    )


thermo_grid = thermo_sweep(
    real_metrics,
    single_site_effects,
    make_thermo,
    alphas=[0.0, 1.0, 1.5, 2.0, 3.0, 5.0],
    scales=[0.3, 0.5, 0.75, 1.0, 1.5],
    depths=DEPTHS,
    n_per_depth=N_PER_DEPTH,
)
best_alpha_thermo, best_scale_thermo = find_optimal_thermo(thermo_grid)
print(f"\nThermo joint best: alpha={best_alpha_thermo}, scale={best_scale_thermo}")
thermo_grid.sort_values("composite_score").head(8).round(4)

  alpha=0.00 scale=0.300

 -> score=0.2431
  alpha=0.00 scale=0.500

 -> score=0.2431
  alpha=0.00 scale=0.750

 -> score=0.2431
  alpha=0.00 scale=1.000

 -> score=0.2431
  alpha=0.00 scale=1.500

 -> score=0.2431
  alpha=1.00 scale=0.300

... (22 outputs left out)

 -> score=0.9886
  alpha=5.00 scale=1.000

 -> score=0.6832
  alpha=5.00 scale=1.500

 -> score=0.3528

Thermo joint best: alpha=1.5, scale=1.0


,alpha,scale,composite_score,additive_corr_d2,sign_epi_d2,additive_corr_d3,sign_epi_d3,additive_corr_d4,sign_epi_d4
13,1.5,1.00,0.1659,0.7749,0.1610,0.5752,0.4417,0.4575,0.5873
19,2.0,1.50,0.1720,0.7843,0.1450,0.5872,0.4353,0.4882,0.5810
7,1.0,0.75,0.1720,0.7843,0.1450,0.5872,0.4353,0.4882,0.5810
24,3.0,1.50,0.1760,0.7378,0.2030,0.5378,0.4777,0.3748,0.6223
18,2.0,1.00,0.1760,0.7378,0.2030,0.5378,0.4777,0.3748,0.6223
12,1.5,0.75,0.1760,0.7378,0.2030,0.5378,0.4777,0.3748,0.6223
6,1.0,0.50,0.1760,0.7378,0.2030,0.5378,0.4777,0.3748,0.6223
14,1.5,1.50,0.1946,0.7984,0.1103,0.6103,0.4190,0.5517,0.5717


In [17]:
# Final held-out comparison: all metrics at the thermo joint best
g_best = GlobalEpistasisSigmoid(floor=_base.floor, ceiling=_base.ceiling, scale=best_scale_thermo)
row_joint = eval_at(
    f"Thermo joint best (alpha={best_alpha_thermo}, scale={best_scale_thermo})",
    best_alpha_thermo,
    g_best,
)
thermo_final = pd.concat(
    [thermo_cmp, pd.DataFrame([row_joint])],
    ignore_index=True,
)
thermo_final.to_csv(OUTPUT_DIR / "gb1_thermo_final.csv", index=False)
thermo_final.round(3)

,config,alpha,add_corr_d2,add_corr_d4,walsh_1,walsh_2,walsh_3,walsh_4,n_optima,accessible_frac,rs_ratio,fdc,tensor_r,walsh_L1,fitness_W1
0,REAL GB1,NaN,0.589,0.141,0.186,0.429,0.308,0.077,182,0.317,0.807,-0.169,NaN,NaN,NaN
1,Identity (baseline),1.5,0.892,0.650,0.841,0.159,0.000,0.000,5,0.772,0.789,-0.372,0.104,1.311,1.323
2,"Thermo, upside=1 (singles-only)",1.5,0.852,0.539,0.668,0.238,0.073,0.021,8,0.738,0.772,-0.387,0.128,0.964,0.547
3,"Thermo, upside=3 (stability prior)",1.5,0.727,0.359,0.457,0.382,0.137,0.024,9,0.729,0.874,-0.322,0.143,0.543,0.417
4,"Thermo joint best (alpha=1.5, scale=1.0)",1.5,0.775,0.458,0.471,0.383,0.129,0.018,4,0.901,0.717,-0.487,0.218,0.571,0.213


## Visual comparison: real vs synthetic

The scalar tables above compress every structural comparison into numbers. The plots below make the same comparisons visible and add three held-out scalar metrics to the per-config table:

- **Sigmoid curves** -- the shape of each fitted $g$. The rug on the left edge shows where the singles-only fit is actually constrained by data vs where it is extrapolating.
- **Walsh spectrum bar chart** -- variance fraction by interaction order, real against every thermo configuration. The story of "does the non-linearity generate the right higher-order variance" told visually.
- **Fitness distribution overlay** -- marginal histograms of real and synthetic fitness values. Shows the dead-variant plateau that a smooth sigmoid cannot reproduce.
- **Per-variant scatter (hexbin)** -- for every variant in the 4-site space, real fitness on the x-axis, synthetic fitness on the y-axis. Pearson r is the scalar; a tight $y = x$ cloud is the picture.

Three matching scalars now live in `thermo_cmp`: **`tensor_r`** (per-variant Pearson r, 1 = perfect prediction), **`walsh_L1`** (sum of $|$real - synth$|$ across Walsh orders, 0 = identical spectra), and **`fitness_W1`** (Wasserstein-1 distance between marginal fitness distributions, 0 = identical shapes). Together with the existing structural metrics they form a defensible quantitative assessment:

| question | metric |
|---|---|
| do we rank variants correctly? | `tensor_r` |
| do we put variance in the right interaction orders? | `walsh_L1` |
| do we produce the right overall fitness shape? | `fitness_W1` |
| do we have the right ruggedness? | `n_optima`, `accessible_frac`, `rs_ratio` |
| do we preserve sign-epistasis and path structure? | `sign_epi_*`, reciprocal sign frac |

In [18]:
# Build a synthetic fitness tensor for each non-linearity configuration at
# alpha=1.5 so the next four plots share the same underlying data.
_thermo_configs = [
    ("Identity", None),
    ("Thermo upside=1", g_singles),
    ("Thermo upside=3", g_upside),
    (f"Thermo joint-best (scale={best_scale_thermo})", g_best),
]
synth_tensors = {}
for label, nl in _thermo_configs:
    ls = build_thermo(1.5, nl)
    synth_tensors[label] = fitness_tensor_from_landscape(ls, POSITIONS, alphabets)

In [19]:
observed_singles_values = np.array(
    [v for effects in single_site_effects.values() for v in effects.values()]
)
fig = fig_sigmoid_curves(
    {
        "singles-only (upside=1)": g_singles,
        "upside=3 (stability prior)": g_upside,
        f"joint-best (scale={best_scale_thermo})": g_best,
    },
    observed_singles_values,
)
save_fig(fig, "gb1_sigmoid_curves", OUTPUT_DIR)

  Saved gb1_sigmoid_curves.pdf/.png


In [20]:
synth_walsh_by_label = {label: walsh_decompose(t) for label, t in synth_tensors.items()}
fig = fig_walsh_spectrum(walsh["real"], synth_walsh_by_label)
save_fig(fig, "gb1_walsh_spectrum", OUTPUT_DIR)

  Saved gb1_walsh_spectrum.pdf/.png


In [21]:
fig = fig_fitness_distribution(real_tensor, synth_tensors)
save_fig(fig, "gb1_fitness_distribution", OUTPUT_DIR)

  Saved gb1_fitness_distribution.pdf/.png


In [22]:
# Per-variant fitness match for the strongest configuration.
_best_label = f"Thermo joint-best (scale={best_scale_thermo})"
fig = fig_tensor_scatter(
    real_tensor,
    synth_tensors[_best_label],
    label=_best_label,
)
save_fig(fig, "gb1_tensor_scatter", OUTPUT_DIR)

  Saved gb1_tensor_scatter.pdf/.png

## Censored sigmoid: pinning the dead-variant plateau

Real GB1 has 76.9% of combinatorial variants at exactly the detection-limit fitness (-1.0). No smooth sigmoid can reproduce that spike -- the asymptote is approached, never pinned. Adding a hard floor to `GlobalEpistasisSigmoid` (`floor_hard` parameter) clips the output at a user-chosen value below zero, producing a censored sigmoid:

$$g_{\text{cens}}(z) = \max\bigl(g_{\text{smooth}}(z),\; \text{floor\_hard}\bigr).$$

The constraint `floor_hard <= min observed single fitness` preserves the single-mutant prediction identity. For GB1 the singles min is -0.9966, so either of {-1.0, -0.9966} is safe.

Expected effect on the four open-gap metrics:
- `fitness_W1` collapses toward zero (spike matches spike).
- `n_optima` climbs from ~9 toward ~100-200 (alive sub-graph fragments; edge-peaks proliferate).
- `accessible_frac` drops toward real's 0.32 (dead plateau blocks strict-uphill paths).
- `reciprocal_sign_frac` rises (doubly-deleterious pairs clip to the floor, producing non-additive rescue-like signatures).

In [23]:
# Censored sigmoid sweep: keep the joint-best (alpha, scale) and add a hard
# floor. Constraint: floor_hard <= min observed single fitness (-0.9966 for
# GB1), otherwise observed singles would be clipped upward and the
# "g(g^-1(h)) = h" identity that pins single-mutant predictions to DMS would
# break. -1.0 is safe; -0.9966 is exactly the singles min.
censored_rows = [thermo_rows[0], thermo_final.iloc[-1].to_dict()]
for fh in [-1.0, -0.9966]:
    g_cens = GlobalEpistasisSigmoid(
        floor=_base.floor,
        ceiling=_base.ceiling,
        scale=best_scale_thermo,
        floor_hard=fh,
    )
    censored_rows.append(eval_at(f"+ floor_hard={fh}", best_alpha_thermo, g_cens))
censored_cmp = pd.DataFrame(censored_rows)
censored_cmp.to_csv(OUTPUT_DIR / "gb1_censored_sweep.csv", index=False)
censored_cmp.round(3)

,config,alpha,add_corr_d2,add_corr_d4,walsh_1,walsh_2,walsh_3,walsh_4,n_optima,accessible_frac,rs_ratio,fdc,tensor_r,walsh_L1,fitness_W1
0,REAL GB1,NaN,0.589,0.141,0.186,0.429,0.308,0.077,182,0.317,0.807,-0.169,NaN,NaN,NaN
1,"Thermo joint best (alpha=1.5, scale=1.0)",1.5,0.775,0.458,0.471,0.383,0.129,0.018,4,0.901,0.717,-0.487,0.218,0.571,0.213
2,+ floor_hard=-1.0,1.5,0.747,0.410,0.398,0.421,0.159,0.023,4,0.634,0.758,-0.463,0.216,0.424,0.116
3,+ floor_hard=-0.9966,1.5,0.747,0.409,0.396,0.421,0.160,0.023,4,0.631,0.759,-0.463,0.216,0.422,0.116


## Experiment 1: noise-as-ruggedness (headline ruggedness parameter)

`noise_sigma` is a scalar parameter on `Landscape` that adds i.i.d. Gaussian noise per variant pre-nonlinearity. It is the paper's primary ruggedness mechanism. The two headline metrics are:

- `walsh_L1`: Walsh spectrum L1 distance to real (structural match)
- `fitness_W1`: Wasserstein-1 distance between fitness distributions (distributional match)

Both return a single float, both have 0 = perfect match. We sweep `noise_sigma ∈ {0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8}` at joint-best thermo + floor_hard=-1.0 and pick the sigma that jointly minimises both scalars.

Expected behaviour: i.i.d. Gaussian noise distributes Walsh variance approximately uniformly across orders, so `walsh_L1` has a shallow minimum (noise closes the order-3/4 gap slightly) and `fitness_W1` stays near its pairwise-only value until noise becomes large enough to inflate the tails.

The sweep also records all SSMuLA navigability attributes (`frac_active`, `frac_local_optima`, `kurtosis`, `kde_peak_count`, `n_optima`, `accessible_frac`) as descriptive, not optimised, to support the "spans the empirical SSMuLA range" coverage claim in the paper.

See `scripts/run_noise_sweep.py` for the standalone version used to produce `output_cross_protein/headline.png`.

In [ ]:
# Headline noise_sigma sweep. Schema matches scripts/run_noise_sweep.py so
# either source produces a CSV consumable by scripts/build_noise_figures.py.
from mutascape.validation.metrics import ssmula_attributes

g_cens = GlobalEpistasisSigmoid(
    floor=_base.floor,
    ceiling=_base.ceiling,
    scale=best_scale_thermo,
    floor_hard=-1.0,
)


def eval_noise(noise_sigma):
    ls = build_landscape(
        DMS_CSV,
        DISTANCES_NPZ,
        WT_FASTA,
        alpha=best_alpha_thermo,
        seed=SEED,
        nonlinearity=g_cens,
        noise_sigma=noise_sigma,
    )
    t = fitness_tensor_from_landscape(ls, POSITIONS, alphabets)
    w = walsh_decompose(t)
    o = local_optima_from_tensor(t)
    p = accessible_paths(t, n_sources=3000, n_rsign_samples=20000, seed=SEED)
    ss = ssmula_attributes(t)
    return {
        "sweep": f"noise={noise_sigma}",
        "noise_sigma": noise_sigma,
        "walsh_1": w[1],
        "walsh_2": w[2],
        "walsh_3": w[3],
        "walsh_4": w[4],
        "walsh_L1": walsh_l1_distance(walsh["real"], w),
        "fitness_W1": fitness_wasserstein(real_tensor, t),
        "n_optima": o["n_optima"],
        "accessible_frac": p["mean_accessible_fraction"],
        "ssmula_frac_active": ss["frac_active"],
        "ssmula_frac_optima": ss["frac_local_optima"],
        "ssmula_kurtosis": ss["kurtosis"],
        "ssmula_kde_peaks": ss["kde_peak_count"],
    }


noise_rows = [eval_noise(s) for s in [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8]]
noise_sweep_df = pd.DataFrame(noise_rows)
noise_sweep_df.to_csv(OUTPUT_DIR / "gb1_noise_sweep.csv", index=False)
noise_sweep_df.round(3)

## Experiment 2: pLM-direct landscape

A completely different signal source: ESM2 (pretrained on ~10⁸ proteins) scores each mutant sequence as an unmasked pseudo-log-likelihood -- sum of `log P(mutant AA at p | full mutant sequence)` over mutated positions. Calibrate linearly to DMS singles. No contact map, no pairwise J, no sigmoid. Just "what does a language model think of this sequence?"

If pLM-direct matches real GB1's Walsh spectrum and ruggedness without the gymnastics of the structural pipeline, it's a simpler generative model for the "synthetic combinatorial distribution" use case. If it matches on some metrics and fails on others, we learn *which* parts of real ruggedness a language model does and doesn't capture.

The tensor is precomputed by `scripts/plm_direct_gb1.py` to keep notebook run time reasonable (full 160k ESM2 forward passes).

In [25]:
# Experiment 2: pLM-direct landscape.
# The tensor is precomputed by scripts/plm_direct_gb1.py (ESM2 unmasked
# pseudo-log-likelihood on every variant, then linear-regression calibration
# against observed DMS singles). Here we just load it and compute the same
# held-out metrics that we computed for the structural + thermo configurations.
PLM_TENSOR_PATH = DATA_DIR / "plm_direct_tensor.npz"

if PLM_TENSOR_PATH.exists():
    _plm = np.load(PLM_TENSOR_PATH, allow_pickle=True)
    plm_tensor = _plm["tensor"]
    plm_a = float(_plm["calibration_a"])
    plm_b = float(_plm["calibration_b"])
    plm_model = str(_plm["model_name"])
    print(f"loaded pLM tensor: shape {plm_tensor.shape}, model {plm_model}")
    print(f"calibration: fitness = {plm_a:.4f} * plm_score + {plm_b:.4f}")

    plm_w = walsh_decompose(plm_tensor)
    plm_opt = local_optima_from_tensor(plm_tensor)
    plm_paths = accessible_paths(plm_tensor, n_sources=3000, n_rsign_samples=20000, seed=SEED)
    plm_row = {
        "config": f"pLM-direct ({plm_model})",
        "alpha": None,
        "walsh_1": plm_w[1],
        "walsh_2": plm_w[2],
        "walsh_3": plm_w[3],
        "walsh_4": plm_w[4],
        "n_optima": plm_opt["n_optima"],
        "accessible_frac": plm_paths["mean_accessible_fraction"],
        "rs_ratio": float("nan"),
        "fdc": fitness_distance_correlation_from_tensor(plm_tensor),
        "tensor_r": tensor_pearson_r(real_tensor, plm_tensor),
        "walsh_L1": walsh_l1_distance(walsh["real"], plm_w),
        "fitness_W1": fitness_wasserstein(real_tensor, plm_tensor),
    }
    plm_cmp = pd.DataFrame([thermo_rows[0], thermo_final.iloc[-1].to_dict(), plm_row])
    plm_cmp.to_csv(OUTPUT_DIR / "gb1_plm_direct_comparison.csv", index=False)
    display(plm_cmp.round(3))

    # Walsh spectrum comparison including pLM-direct
    fig = fig_walsh_spectrum(
        walsh["real"],
        {**synth_walsh_by_label, "pLM-direct": plm_w},
    )
    save_fig(fig, "gb1_walsh_spectrum_with_plm", OUTPUT_DIR)

    # Fitness distribution with pLM-direct overlay
    fig = fig_fitness_distribution(
        real_tensor,
        {**synth_tensors, "pLM-direct": plm_tensor},
    )
    save_fig(fig, "gb1_fitness_distribution_with_plm", OUTPUT_DIR)

    # Per-variant scatter: pLM predictions vs real
    fig = fig_tensor_scatter(real_tensor, plm_tensor, label="pLM-direct")
    save_fig(fig, "gb1_tensor_scatter_plm", OUTPUT_DIR)
else:
    print(
        f"pLM tensor not found at {PLM_TENSOR_PATH}. "
        "Run `uv run python scripts/plm_direct_gb1.py` first."
    )

loaded pLM tensor: shape (20, 20, 20, 20), model esm2_t6_8M_UR50D
calibration: fitness = -0.2890 * plm_score + -0.6114


,config,alpha,add_corr_d2,add_corr_d4,walsh_1,walsh_2,walsh_3,walsh_4,n_optima,accessible_frac,rs_ratio,fdc,tensor_r,walsh_L1,fitness_W1
0,REAL GB1,NaN,0.589,0.141,0.186,0.429,0.308,0.077,182,0.317,0.807,-0.169,NaN,NaN,NaN
1,"Thermo joint best (alpha=1.5, scale=1.0)",1.5,0.775,0.458,0.471,0.383,0.129,0.018,4,0.901,0.717,-0.487,0.218,0.571,0.213
2,pLM-direct (esm2_t6_8M_UR50D),NaN,NaN,NaN,0.844,0.145,0.011,0.000,8,0.841,NaN,-0.428,0.006,1.317,1.195


  Saved gb1_walsh_spectrum_with_plm.pdf/.png


  Saved gb1_fitness_distribution_with_plm.pdf/.png
  Saved gb1_tensor_scatter_plm.pdf/.png


## Experimental alternative: structured higher-order couplings (Contact-NK)

The noise sweep above moves `walsh_L1` and `fitness_W1` sensibly and covers a broad SSMuLA attribute range, but it distributes Walsh variance approximately uniformly across orders. It does not preferentially add variance at orders 3+. `ContactNKEpistasis` is an alternative parameter that does: per-position random functions over (position + structural neighbours), pinned so the all-wildtype state is zero. Each function contributes variance at all Walsh orders from 1 to 1 + K_i.

**Known limitation documented below:** at K=3 the Contact-NK random tensors concentrate variance at the maximum order (order 4), overshooting it while under-producing order 3. This is why Contact-NK is demoted to an experimental alternative rather than the headline mechanism.

For GB1's 4 positions, the contact graph has 2 triangles (39-40-41 and 39-41-54) and no 4-cliques (40-54 are just over the 8 A threshold). Neighbours per position: {39: (40,41,54), 40: (39,41), 41: (39,40,54), 54: (39,41)}. Maximum K = 3, so NK functions reach Walsh order 4.

Running the full sweep below at joint-best thermo + floor_hard=-1.0 for reference and future work.

In [26]:
# Contact-NK sweep at joint-best thermo + floor_hard=-1.0.
# sigma_nk is the new ruggedness parameter; alpha_ho=1.0 by default.
g_cens = GlobalEpistasisSigmoid(
    floor=_base.floor,
    ceiling=_base.ceiling,
    scale=best_scale_thermo,
    floor_hard=-1.0,
)


def eval_nk(sigma_nk):
    ls = build_landscape(
        DMS_CSV,
        DISTANCES_NPZ,
        WT_FASTA,
        alpha=best_alpha_thermo,
        seed=SEED,
        nonlinearity=g_cens,
        sigma_nk=sigma_nk,
        alpha_ho=1.0,
    )
    t = fitness_tensor_from_landscape(ls, POSITIONS, alphabets)
    w = walsh_decompose(t)
    o = local_optima_from_tensor(t)
    p = accessible_paths(t, n_sources=3000, n_rsign_samples=20000, seed=SEED)
    ss = ssmula_attributes(t)
    return {
        "config": f"contact-NK sigma_nk={sigma_nk}",
        "sigma_nk": sigma_nk,
        "walsh_1": w[1],
        "walsh_2": w[2],
        "walsh_3": w[3],
        "walsh_4": w[4],
        "n_optima": o["n_optima"],
        "accessible_frac": p["mean_accessible_fraction"],
        "rse_fraction": p["reciprocal_sign_frac"],
        "fdc": fitness_distance_correlation_from_tensor(t),
        "tensor_r": tensor_pearson_r(real_tensor, t),
        "walsh_L1": walsh_l1_distance(walsh["real"], w),
        "fitness_W1": fitness_wasserstein(real_tensor, t),
        "ssmula_frac_active": ss["frac_active"],
        "ssmula_frac_optima": ss["frac_local_optima"],
        "ssmula_kurtosis": ss["kurtosis"],
        "ssmula_kde_peaks": ss["kde_peak_count"],
    }


real_ss = ssmula_attributes(real_tensor)
real_nk_row = {
    "config": "REAL GB1",
    "sigma_nk": None,
    "walsh_1": walsh["real"][1],
    "walsh_2": walsh["real"][2],
    "walsh_3": walsh["real"][3],
    "walsh_4": walsh["real"][4],
    "n_optima": real_opt["n_optima"],
    "accessible_frac": real_paths["mean_accessible_fraction"],
    "rse_fraction": real_paths["reciprocal_sign_frac"],
    "fdc": fitness_distance_correlation_from_tensor(real_tensor),
    "tensor_r": float("nan"),
    "walsh_L1": float("nan"),
    "fitness_W1": float("nan"),
    "ssmula_frac_active": real_ss["frac_active"],
    "ssmula_frac_optima": real_ss["frac_local_optima"],
    "ssmula_kurtosis": real_ss["kurtosis"],
    "ssmula_kde_peaks": real_ss["kde_peak_count"],
}

nk_rows = [real_nk_row] + [eval_nk(s) for s in [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]]
nk_sweep_df = pd.DataFrame(nk_rows)
nk_sweep_df.to_csv(OUTPUT_DIR / "gb1_contact_nk_sweep.csv", index=False)
nk_sweep_df.round(3)

,config,sigma_nk,walsh_1,walsh_2,walsh_3,walsh_4,n_optima,accessible_frac,rse_fraction,fdc,tensor_r,walsh_L1,fitness_W1,ssmula_frac_active,ssmula_frac_optima,ssmula_kurtosis,ssmula_kde_peaks
0,REAL GB1,NaN,0.186,0.429,0.308,0.077,182,0.317,0.152,-0.169,NaN,NaN,NaN,0.024,0.001,76.923,32.0
1,contact-NK sigma_nk=0.0,0.00,0.398,0.421,0.159,0.023,4,0.634,0.007,-0.463,0.216,0.424,0.116,0.057,0.000,40.385,21.0
2,contact-NK sigma_nk=0.1,0.10,0.416,0.404,0.148,0.032,5,0.661,0.011,-0.468,0.215,0.461,0.159,0.070,0.000,33.542,16.0
3,contact-NK sigma_nk=0.25,0.25,0.424,0.350,0.148,0.078,23,0.648,0.029,-0.460,0.205,0.479,0.253,0.098,0.000,24.412,13.0
4,contact-NK sigma_nk=0.5,0.50,0.396,0.239,0.183,0.182,144,0.405,0.074,-0.256,0.175,0.631,0.515,0.167,0.001,12.702,4.0
5,contact-NK sigma_nk=0.75,0.75,0.343,0.159,0.233,0.266,340,0.334,0.128,-0.234,0.142,0.692,0.926,0.253,0.002,5.872,1.0
6,contact-NK sigma_nk=1.0,1.00,0.284,0.115,0.278,0.322,565,0.238,0.171,-0.174,0.115,0.688,1.461,0.340,0.004,2.268,3.0


### Thermo findings

Identity baseline vs Thermo upside=3 (at the same alpha=1.5):

- Composite validation score: `0.253 -> 0.19` (25% improvement).
- Walsh order 3 variance: `0 -> 0.137` (45% of real's 0.308). Walsh order 4: `0 -> 0.024`. These were zero by construction under the pairwise-only model; the non-linearity genuinely manufactures higher-order Walsh variance from a pairwise-additive underlying model, as Sailer & Harms 2017 predicted.
- Additive correlation at depth 2: `0.89 -> 0.73` (real 0.59). Depth 4: `0.65 -> 0.36` (real 0.14). Much closer.
- r/s ratio: `0.79 -> 0.87` (real 0.81). Within 8%.
- Local optima count: `5 -> 9`. Moving, but still far from real's 182 — the dead-variant plateau effect is not captured by a smooth sigmoid alone.
- Accessible path fraction: only marginal improvement (`0.77 -> 0.73`, real 0.31). Another indication that dead-floor structure matters for path metrics.

The composite-joint-best picks a gentler sigmoid (scale=1.0 vs MLE 0.71) that lowers composite further (to 0.166) but sacrifices some structural-metric match. The upside=3 MLE-scale configuration is the pragmatic winner for no-DMS use: every parameter was fit from singles alone plus a single stability-stacking prior.

The central improvement needed from here is a better model for the dead-variant plateau (either a hard floor or an experimental-noise model), which would lift local-optima counts toward real without having to add higher-order couplings.

## What this notebook demonstrates

The notebook follows a **calibrate on low-order, verify on high-order** proof structure. None of the held-out metrics enter the calibration objective, so their agreement (or disagreement) is an honest out-of-objective test of mutascape's behaviour.

**Step 1 - calibrate $\alpha$ on low-order epistasis (cells "Coarse alpha sweep" through the refine cell).** We search $\alpha$ against a composite objective built from (a) Pearson correlation between the additive prediction and actual fitness at depths 2/3/4, (b) sign-epistasis fraction at each depth, (c) Wasserstein distance between real and synthetic epistatic-deviation distributions, (d) fitness-mean RMSE by depth. This objective only sees single- and pairwise-scale statistics; it knows nothing about 3rd/4th-order interactions, local optima, or path accessibility. Coarse+refine picks $\alpha = 1.5$.

**Step 2 - freeze $\alpha$, evaluate held-out structural properties (cells "Complete-space statistics" onwards).** At $\alpha = 1.5$ we build complete $20 \times 20 \times 20 \times 20$ fitness tensors for both the real and synthetic landscapes, then compute the Walsh decomposition, local-optima count, accessible-path fraction, reciprocal sign-epistasis fraction, tensor-level FDC, and the landscape-wide $r/s$ ratio. None of these were available to the calibration.

**Step 3 - compare honestly (summary table and ablation).** The summary table places real and synthetic values side by side with absolute differences. The `RandomEpistasis` vs `SignConsistentEpistasis` ablation tests whether the internal structure of the epistatic couplings (not just their strength) changes the held-out fit.

### What counts as "the setup working"?

The setup works in a limited but precise sense:

- **Reproduced.** The $r/s$ ratio matches real GB1 within ~3% (real 0.807, synth 0.789). The sign of the fitness-distance correlation agrees (both negative). The sign-epistasis profile across depths is tracked closely (depth 4: real 0.658, synth 0.667). None of these are tuned targets -- $r/s$ in particular is a scalar summary of landscape ruggedness that agrees without anyone asking it to.
- **Not reproduced.** Walsh orders 3 and 4 hold ~39% of real GB1's fitness variance but are zero by construction in a pairwise-only coupling model. Local-optima count is ~36x lower in the synthetic landscape (5 vs 182), and the accessible-path fraction is more than double the real value (0.77 vs 0.32). The current model is genuinely smoother and more navigable than real GB1.

The contribution is therefore twofold: mutascape's pairwise calibration *does* capture the gross shape of landscape roughness, and the specific places where it fails point to the missing modelling ingredient -- explicit higher-order couplings. Both halves are necessary for a defensible validation.